In [ ]:
from google.colab import files
files.upload()

In [ ]:
!pip install gensim

In [3]:
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from gensim.models import FastText
from collections import Counter
from torch.utils.data import Dataset , DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [4]:
train_df = pd.read_csv("train.csv" , header=0)
test_df = pd.read_csv("test.csv" , header = 0)

In [5]:
train_df = train_df.iloc[:, :3]
test_df = test_df.iloc[:, :3]

train_df.columns = ["Class", "Title", "Description"]
test_df.columns = ["Class", "Title", "Description"]

train_df = train_df.dropna()
test_df = test_df.dropna()

train_df["Text"] = train_df["Title"] + " " + train_df["Description"]
test_df["Text"] = test_df["Title"] + " " + test_df["Description"]


def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = " ".join(text.split())
    return text

train_df['Text'] = train_df['Text'].apply(clean_text)
test_df['Text'] = test_df['Text'].apply(clean_text)

print("Class DISTRIBUTION")
train_df['Class'] = train_df['Class'].astype(int)
test_df['Class'] = test_df['Class'].astype(int)
class_mapping = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}

train_dist = train_df['Class'].value_counts().sort_index()
test_dist = test_df['Class'].value_counts().sort_index()
print("TRAIN:")
for class_id, count in train_dist.items():
    percentage = (count / len(train_df)) * 100
    print(f"      Class {class_id} ({class_mapping[class_id]}): {count:6d} ({percentage:5.2f}%)")


print("TEXT LENGTH ANALYSIS")
train_lengths = train_df['Text'].str.split().str.len()
test_lengths = test_df['Text'].str.split().str.len()
print(f"   TRAIN:")
print(f"      Min: {train_lengths.min():5d} words")
print(f"      Max: {train_lengths.max():5d} words")
print(f"      Mean: {train_lengths.mean():7.2f} words")
print(f"      Median: {train_lengths.median():5.0f} words")

print(f"   TEST:")
print(f"      Min: {test_lengths.min():5d} words")
print(f"      Max: {test_lengths.max():5d} words")
print(f"      Mean: {test_lengths.mean():7.2f} words")
print(f"      Median: {test_lengths.median():5.0f} words")

train_word_count = Counter()
for text in train_df['Text']:
    train_word_count.update(text.split())

print(f"\n5️⃣ VOCABULARY:")
print(f"   Unique words in train: {len(train_word_count):,}")
print(f"   Top 10 words:")
for word, count in train_word_count.most_common(10):
    print(f"      '{word}': {count:6d} times")

print(f"Columns of Training Dataset:{train_df.columns}")
print(f"Columns of Test Dataset:{test_df.columns}")
print(f"Shape of Traning Dataset:{train_df.shape}")
print(f"Shape of Test Dataset:{test_df.shape}")
print(f"Number of Nan in Training Dataset:{train_df.isnull().sum().sum()}")
print(f"Number of Nan in Test Dataset:{test_df.isnull().sum().sum()}")
print(f"Number of Duplicates in Training Dataset:{train_df.duplicated().sum()}")
print(f"Number of Duplicates in Test Dataset:{test_df.duplicated().sum()}")
print(f"lenght of Training Dataset:{len(train_df)}")
print(f"lenght of Test Dataset:{len(test_df)}")

Class DISTRIBUTION
TRAIN:
      Class 1 (World):  30000 (25.00%)
      Class 2 (Sports):  30000 (25.00%)
      Class 3 (Business):  30000 (25.00%)
      Class 4 (Sci/Tech):  30000 (25.00%)
TEXT LENGTH ANALYSIS
   TRAIN:
      Min:     4 words
      Max:   177 words
      Mean:   37.41 words
      Median:    37 words
   TEST:
      Min:    11 words
      Max:   136 words
      Mean:   37.29 words
      Median:    37 words

5️⃣ VOCABULARY:
   Unique words in train: 102,157
   Top 10 words:
      'the': 203517 times
      'to': 119014 times
      'a': 107540 times
      'of':  97903 times
      'in':  95429 times
      'and':  68842 times
      'on':  56500 times
      'for':  50168 times
      '39s':  31218 times
      'that':  27740 times
Columns of Training Dataset:Index(['Class', 'Title', 'Description', 'Text'], dtype='object')
Columns of Test Dataset:Index(['Class', 'Title', 'Description', 'Text'], dtype='object')
Shape of Traning Dataset:(120000, 4)
Shape of Test Dataset:(7600, 4)
N

#EMBEDDING

In [7]:
train_texts = [text.split() for text in train_df["Text"]]
test_texts = [text.split() for text in test_df["Text"]]

print(f"Train texts: {len(train_texts)} samples")
print(f"Test texts: {len(test_texts)} samples")

print("Training FastText Models")

ft_model = FastText(
    sentences=train_texts,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10,
    seed=42
)

print("✅ FastText model trained!")
print(f"Vocabulary size: {len(ft_model.wv)}")

Train texts: 120000 samples
Test texts: 7600 samples
Training FastText Models
✅ FastText model trained!
Vocabulary size: 54850


#MATRIX & VOCAB ENCODE

In [8]:
class VocabWithEmbedding:
  def __init__(self , texts , embedding_model):
    self.word2idx = {"<PAD>" : 0, "<UNK>" : 1}
    self.emdedding = embedding_model

    word_count = Counter()
    for text in texts:
      word_count.update(text)

    idx = 2
    for word , count in word_count.items():
      if count >= 2:
        self.word2idx[word] = idx
        idx += 1
  def encode(self,text):
    if isinstance(text , str):
      tokens = text.split()
    else:
      tokens = text
    return [self.word2idx.get(token, 1) for token in tokens]
  def get_embedding_matrix(self):
    embedding_matrix = np.zeros((len(self.word2idx), 100))
    embedding_matrix[0] = np.zeros(100)  # PAD token
    embedding_matrix[1] = np.random.randn(100)  # UNK token

    for word, idx in self.word2idx.items():
        if word not in ['<PAD>', '<UNK>']:
            try:
                embedding_matrix[idx] = ft_model.wv[word]
            except KeyError:
                embedding_matrix[idx] = np.random.randn(100)

    return torch.tensor(embedding_matrix, dtype=torch.float32)

print("BUILDING VOCABULARY")
print("=" * 70)

vocab = VocabWithEmbedding(train_texts, ft_model)
embedding_matrix = vocab.get_embedding_matrix()

print(f"\n Vocabulary size: {len(vocab.word2idx)}")
print(f" Embedding matrix shape: {embedding_matrix.shape}")
print(f"   - Tokens: {embedding_matrix.shape[0]}")
print(f"   - Dimensions: {embedding_matrix.shape[1]}")

# ============ ENCODE TEXTS ============

print(" ENCODING TEXTS")
print("=" * 70)

train_df['encoded'] = train_df['Text'].apply(lambda x: vocab.encode(x))
test_df['encoded'] = test_df['Text'].apply(lambda x: vocab.encode(x))

print(f"\nTrain texts encoded: {len(train_df)}")
print(f" Test texts encoded: {len(test_df)}")

BUILDING VOCABULARY

 Vocabulary size: 54852
 Embedding matrix shape: torch.Size([54852, 100])
   - Tokens: 54852
   - Dimensions: 100
 ENCODING TEXTS

Train texts encoded: 120000
 Test texts encoded: 7600


In [9]:
import torch
from torch.nn.utils.rnn import pad_sequence

print("=" * 70)
print("📊 PREPARING DATA")
print("=" * 70)

train_sequences = [torch.tensor(seq, dtype=torch.long) for seq in train_df['encoded']]
test_sequences = [torch.tensor(seq, dtype=torch.long) for seq in test_df['encoded']]

X_train = pad_sequence(train_sequences, batch_first=True, padding_value=0)
X_test = pad_sequence(test_sequences, batch_first=True, padding_value=0)

y_train = torch.tensor((train_df['Class'].values - 1), dtype=torch.long)
y_test = torch.tensor((test_df['Class'].values - 1), dtype=torch.long)

print(f"\n✅ X_train shape: {X_train.shape}")
print(f"✅ y_train shape: {y_train.shape}")
print(f"✅ X_test shape: {X_test.shape}")
print(f"✅ y_test shape: {y_test.shape}")

max_len = X_train.shape[1]
print(f"✅ Max sequence length: {max_len}")

📊 PREPARING DATA

✅ X_train shape: torch.Size([120000, 177])
✅ y_train shape: torch.Size([120000])
✅ X_test shape: torch.Size([7600, 136])
✅ y_test shape: torch.Size([7600])
✅ Max sequence length: 177


# MAKING DEEP LEARNING MODEL

**MODEL NAME  = LSTM**

In [10]:
class TextClassifier(nn.Module):
  def __init__(self , embedding_matrix , hidden_dim = 128 ,num_classes=4):
    super(TextClassifier , self).__init__()

    vocab_size , embedding_dim = embedding_matrix.shape

    self.embedding = nn.Embedding(vocab_size , embedding_dim , padding_idx=0)
    self.embedding.weight.data.copy_(embedding_matrix)
    self.embedding.weight.requires_grad = False
    self.lstm = nn.LSTM(
        input_size=embedding_dim,
        hidden_size=hidden_dim,
        num_layers=1,
        batch_first=True,
        bidirectional= True
    )
    self.attention = nn.Linear(hidden_dim * 2, 1)
    self.fc1 = nn.Linear(hidden_dim * 2, 64)
    self.dropout = nn.Dropout(0.5)
    self.fc2 = nn.Linear(64, num_classes)
  def forward(self , text):
    embedded = self.embedding(text)
    lstm_out, _ = self.lstm(embedded)
    attention_weights = torch.tanh(self.attention(lstm_out))  # (batch_size, seq_len, 1)
    attention_weights = torch.softmax(attention_weights, dim=1)
    context = torch.sum(lstm_out * attention_weights, dim=1)

    x = self.fc1(context)
    x = torch.relu(x)
    x = self.dropout(x)
    x = self.fc2(x)

    return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n Device: {device}")
model = TextClassifier(embedding_matrix, hidden_dim=128, num_classes=4)
model.to(device)

print(f" Model initialized")
print(f"\n{model}")


 Device: cuda
 Model initialized

TextClassifier(
  (embedding): Embedding(54852, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True, bidirectional=True)
  (attention): Linear(in_features=256, out_features=1, bias=True)
  (fc1): Linear(in_features=256, out_features=64, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=64, out_features=4, bias=True)
)


In [11]:
import sys
import torch.optim as optim
from sklearn.metrics import accuracy_score , f1_score  , confusion_matrix
from tqdm.notebook import tqdm

print("="*35)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters() , lr=0.001)
print("Loss Function : CrossEntropyLoss")
print("Optimizer : ADAM ")
print("\n" + "=" * 35)
print("TRAINING")
print("=" * 35)
num_epochs =10
batch_size = 8

for epoch in range(num_epochs):
  model.train()
  total_loss = 0
  correct = 0
  total = 0

  num_batches = (len(X_train) + batch_size - 1) // batch_size
  pbar = tqdm(range(0, len(X_train), batch_size),
              desc=f'Epoch [{epoch+1}/{num_epochs}]',
              total=num_batches)

  for i in pbar:
    batch_X = X_train[i:i + batch_size].to(device)
    batch_y = y_train[i:i + batch_size].to(device)
    #Forward pass
    outputs = model(batch_X)
    loss = criterion(outputs , batch_y)

    #Back - Propagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Statistics
    total_loss += loss.item()
    _, predicted = torch.max(outputs, 1)
    correct += (predicted == batch_y).sum().item()
    total += batch_y.size(0)
  avg_loss = total_loss / ((len(X_train) // batch_size) + 1)
  accuracy = correct / total * 100

  print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")


Loss Function : CrossEntropyLoss
Optimizer : ADAM 

TRAINING


Epoch [1/10]:   0%|          | 0/15000 [00:00<?, ?it/s]

Epoch [1/10] Loss: 0.3410, Accuracy: 88.60%


Epoch [2/10]:   0%|          | 0/15000 [00:00<?, ?it/s]

Epoch [2/10] Loss: 0.2583, Accuracy: 91.24%


...

Epoch [8/10] Loss: 0.1556, Accuracy: 94.29%


Epoch [9/10]:   0%|          | 0/15000 [00:00<?, ?it/s]

Epoch [9/10] Loss: 0.1464, Accuracy: 94.53%


Epoch [10/10]:   0%|          | 0/15000 [00:00<?, ?it/s]

Epoch [10/10] Loss: 0.1406, Accuracy: 94.61%


In [12]:
model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for i in range(0, len(X_test), batch_size):
        batch_X = X_test[i:i + batch_size].to(device)
        batch_y = y_test[i:i + batch_size].to(device)

        outputs = model(batch_X)
        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())
accuracy = accuracy_score(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions, average='weighted')
cm = confusion_matrix(all_labels, all_predictions)

print(f"\n📊 TEST RESULTS:")
print(f"   Accuracy: {accuracy*100:.2f}%")
print(f"   F1-Score: {f1*100:.2f}%")

print(f"\n📋 Confusion Matrix:")
class_names = ['World', 'Sports', 'Business', 'Sci/Tech']
for i, class_name in enumerate(class_names):
    print(f"   {class_name}: {cm[i]}")



📊 TEST RESULTS:
   Accuracy: 91.12%
   F1-Score: 91.10%

📋 Confusion Matrix:
   World: [1722   51   71   56]
   Sports: [  22 1857    9   12]
   Business: [  68   17 1663  152]
   Sci/Tech: [  58   16  143 1683]


In [13]:
import torch
import pickle


with open("vocab.pkl" , 'wb') as f:
  pickle.dump(vocab.word2idx , f)
print("✅ Vocabulary saved as vocab.pkl")

checkpoint = {
    'model_state_dict': model.state_dict(),
    'embedding_matrix': embedding_matrix,
    'vocab': vocab.word2idx,
    'vocab_size': len(vocab.word2idx),
    'embedding_dim': 100,
    'hidden_dim': 128,
    'num_classes': 4,
    'max_sequence_length': X_train.shape[1],
    'class_mapping': {
        0: 'World',
        1: 'Sports',
        2: 'Business',
        3: 'Sci/Tech'
    }
}

torch.save(checkpoint, 'complete_checkpoint.pth')
print("✅ Complete checkpoint saved as complete_checkpoint.pth")

torch.save(model.state_dict() , "model_weights.pth")
ft_model.save("fasttext_model.bin")

✅ Vocabulary saved as vocab.pkl
✅ Complete checkpoint saved as complete_checkpoint.pth


In [ ]:
from google.collab import files

print("\n📥 Downloading all necessary files...\n")

files.download('model_weights.pth')
print("✅ Downloaded: model_weights.pth")

files.download('fasttext_model.bin')
print("✅ Downloaded: fasttext_model.bin")

files.download('vocab.pkl')
print("✅ Downloaded: vocab.pkl")

files.download('complete_checkpoint.pth')
print("✅ Downloaded: complete_checkpoint.pth")